In [125]:
import pandas as pd
import numpy as np


##Importing and Cleaning Dataframes

In [126]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Reading in the webscraped dataframes from csv in our drive and cleaning**

In [127]:
df_rankings = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/team_rankings.csv')

In [128]:
df_articles = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/Newest Copy of articles.csv')

In [129]:
df_rosters = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/df_rosters.csv')

In [130]:
df_team_mentions = pd.read_csv('drive/MyDrive/Colab Notebooks/Data Project/dataframes/TeamMentions.csv')

Removing problematic strings for analysis:

In [131]:
df_articles["text"] = df_articles["text"]
df_articles["text"] = df_articles["text"].str.replace("\n", "")

# List of teams to remove since they are similar to ones we want
removable_words = ["UMass-Lowell", 'Carleton University', "Texas A&M", "Texas State", "Texas-San Antonio", "North Texas", "South Texas"
,"CaliforniaState", "Southern California", "NorCal", "Washington State", "Western Washington", "George Washington", "Minnesota Duluth", "UNCC"]

pattern = '|'.join(rf'\b{word}\b' for word in removable_words)
df_articles['text'] = df_articles['text'].str.replace(pattern, '', case=False)

<ipython-input-131-2a4a67cb7584>:9: FutureWarning:

The default value of regex will change from True to False in a future version.



Cleaning the Team Names in the roster dataframe

In [132]:
team_names = {}

def get_team_name(x):
  names = x.split('(')
  return names[1].replace(')', '')

def f_the_paren(x):
  return x.split('(')[0]

df_rosters['Team Name'] = df_rosters['Team'].map(get_team_name)
df_rosters['Team'] = df_rosters['Team'].map(f_the_paren)


Populating a dictionary with every way a team is mentioned.



*   Key: (STR) the team name as specified in df_rosters
*   Value: (ARRAY) a list of every known name, moniker, school, and player with which a team can be mentioned





In [133]:
team_names = {}
for team, team_name in zip(df_rosters['Team'], df_rosters["Team Name"]):
  team = team.strip(' ')
  team_name = team_name.strip(' ')

  if team not in team_names:
    team_names[team] = [team_name]

for team in df_rosters['Team']:
  team = team.strip(' ')
  if team not in team_names[team]:
    team_names[team].append(team)

#Adding each player's FULL names to the dictionary per team
for team in df_rosters['Team']:
  team = team.strip(' ')
  if team not in team_names[team]:
    team_names[team].append(team)

##TF-IDF

Computing tf-idf for every article:

In [134]:
from collections import Counter

bag_of_words = (
    df_articles["text"].
    str.lower().
    str.replace("[^\w\s]", " ").
    str.split()
).apply(Counter)

tf = pd.DataFrame(list(bag_of_words))
tf = tf.fillna(0)
df = (tf > 0).sum(axis=0)
idf = np.log(len(tf) / df)
tf_idf = tf * idf
tf_idf.head()

<ipython-input-134-968f5e6f9445>:6: FutureWarning:

The default value of regex will change from True to False in a future version.



,recognizing,the,top,seven,performers,of,2023,season,june,12,...,caleb,seamon,unsanctioned,tori,gray,await,unmet,shockers,longshots,reaches
0,3.988984,0.0,8.217227,10.090588,8.788898,0.0,0.0,8.206166,3.701302,2.652792,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3.988984,0.0,1.027153,2.522647,0.000000,0.0,0.0,2.564427,3.701302,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,6.162920,0.000000,0.000000,0.0,0.0,0.512885,3.701302,1.326396,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,1.025771,3.701302,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,2.051541,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##Using Team mentions to for a KNN Classifier


In [135]:
training_set = df_team_mentions.drop(['author', 'text', 'title', 'date'], axis = 1)
y_train = df_articles["author"]

In [136]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=10)
)

# GridSearchCV will replace n_neighbors by values in param_grid
grid_search = GridSearchCV(pipeline,
                           param_grid={
                               "kneighborsclassifier__n_neighbors": range(1, 20)
                           },
                           scoring="accuracy",
                           cv=5)

grid_search.fit(training_set, y_train)

df_cv_results = pd.DataFrame(grid_search.cv_results_)

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning:

The least populated class in y has only 1 members, which is less than n_splits=5.



In [137]:
df_cv_results["param_kneighborsclassifier__n_neighbors"] = df_cv_results["param_kneighborsclassifier__n_neighbors"].astype(float)

df_cv_results.set_index("param_kneighborsclassifier__n_neighbors", inplace = True)

In [138]:
import plotly.express as px
fig = px.line(df_cv_results,  y = "mean_test_score")
fig.update_layout(yaxis_title='Accruacy', xaxis_title = "Neighbors", width=900, height=800,
                  yaxis_tickfont=dict(size=16), xaxis_title_font=dict(size=20), yaxis_title_font=dict(size=20), title_font = dict(family= "Impact, sans-serif", size = 28), title = "kNN Models with team mentions as features")
fig.show()

In [140]:
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=14)
)

pipeline.fit(X=training_set, y=y_train)
y_train_ = pd.Series(pipeline.predict(training_set), name = "Predicted")
print(accuracy_score(y_train, y_train_))

0.5


50% accuracy is a slightly better result than we had with tf-idf perdictions. Lets look at which articles the pipline predicted for each author:

In [141]:
y_train_ = pd.Series(pipeline.predict(training_set), name = "Predicted")

pd.crosstab(y_train, y_train_, margins=True)

Predicted,AIDAN SHAPIRO-LEIGHTON,ALEX RUBIN,CHARLIE EISENHOOD,EDWARD STEPHENS,JAKE THORNE,JESSE STROD,KEITH RAYNOR,ULTIWORLD,All
author,,,,,,,,,
AIDAN SHAPIRO-LEIGHTON,2,0,0,0,0,0,10,0,12
ALEX RUBIN,0,1,0,5,0,0,3,0,9
BRAD WARD,0,0,0,2,0,0,0,0,2
BRIDGET MIZENER,0,0,1,0,0,0,1,0,2
CHARLIE EISENHOOD,0,0,1,0,0,0,9,1,11
CHRIS CASSELLA,0,0,0,1,0,0,1,0,2
DANIE PROBY,0,0,0,0,0,0,1,0,1
EDWARD STEPHENS,0,1,2,11,0,1,9,0,24
"FIONA ""SCOTTI"" NUGENT",0,0,0,1,0,0,0,0,1


While are model is still predicting Keith Raynor to write the majorioty of the articles, using team mentions as our features resulted in a healthier spread of predictions. At the every least, our model did not miss a keith raynor article.

##Using TF-IDF for KNN Classifier
Now we attempt using tf-idf to train a K nearest neighbors classification model to identify the author of the text.

In [142]:
x_train = tf_idf
y_train = df_articles["author"]

Using hyper parameter tuning to find best k to predict author:

In [112]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=10)
)

# GridSearchCV will replace n_neighbors by values in param_grid
grid_search = GridSearchCV(pipeline,
                           param_grid={
                               "kneighborsclassifier__n_neighbors": range(1, 20)
                           },
                           scoring="accuracy",
                           cv=5)

grid_search.fit(x_train, y_train)

df_cv_results = pd.DataFrame(grid_search.cv_results_)


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning:

The least populated class in y has only 1 members, which is less than n_splits=5.



In [113]:
df_cv_results["param_kneighborsclassifier__n_neighbors"] = df_cv_results["param_kneighborsclassifier__n_neighbors"].astype(float)

df_cv_results.set_index("param_kneighborsclassifier__n_neighbors", inplace = True)

In [114]:
fig = px.line(df_cv_results,  y = "mean_test_score")
fig.update_layout(yaxis_title='Accruacy', xaxis_title = "Neighbors", width=900, height=800,
                  yaxis_tickfont=dict(size=16), xaxis_title_font=dict(size=20), yaxis_title_font=dict(size=20), title_font = dict(family= "Impact, sans-serif", size = 28), title = "kNN Models with tf idf as Features")
fig.show()

We see that are best k value is 4, and our best accuracy using cross valdiation is under 40%. Lets implement that particular model and see what we can learn:

In [116]:
from sklearn.metrics import accuracy_score
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=4)
)

pipeline.fit(X=x_train, y=y_train)
y_train_ = pd.Series(pipeline.predict(x_train), name = "Predicted")
print(accuracy_score(y_train, y_train_))

0.42592592592592593


In [117]:
y_train_ = pd.Series(pipeline.predict(x_train), name = "Predicted")

pd.crosstab(y_train, y_train_, margins=True)

Predicted,CHARLIE EISENHOOD,KEITH RAYNOR,All
author,,,
AIDAN SHAPIRO-LEIGHTON,0,12,12
ALEX RUBIN,0,9,9
BRAD WARD,0,2,2
BRIDGET MIZENER,0,2,2
CHARLIE EISENHOOD,9,2,11
CHRIS CASSELLA,0,2,2
DANIE PROBY,1,0,1
EDWARD STEPHENS,1,23,24
"FIONA ""SCOTTI"" NUGENT",0,1,1


Of the 18 possible authors our model only predicted two of them to have written any of the articles. Unfortunately, I don't think we are going to predict which author wrote a particular article based on tf idf and knn classification, although it was an interesting attempt.


##Computing Cosine Distance to find articles in which SLOCORE is mentioned

In [118]:
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

tf_idf_similarity = cosine_similarity(tf_idf)

Finding an article that mentions slocore a lot:

In [119]:
df_articles.loc[28] # this article mentions all 20 teams, and mentions slocore a lot as well.
df_team_mentions.loc[28]

Unnamed: 0                                                              28
author                                                         JAKE THORNE
text                     Get to know the 20 teams competing for a D-I c...
date                                                          May 25, 2023
title                    D-I College Championships 2023: Pool Previews ...
North Carolina                                                        22.0
Massachusetts                                                         12.0
Cal Poly-SLO                                                          18.0
Colorado                                                              26.0
Vermont                                                               13.0
Oregon                                                                13.0
Brown                                                                 14.0
Pittsburgh                                                            13.0
California-Santa Cruz    

lets compare that article to all the others with its cosine distance:

In [120]:
df_articles['similarity to 28'] = tf_idf_similarity[28]

In [121]:
df_articles.sort_values(by='similarity to 28', ascending=False).head(n=5)

,author,text,date,title,similarity to 28
28,JAKE THORNE,Get to know the 20 teams competing for a D-I c...,"May 25, 2023",D-I College Championships 2023: Pool Previews ...,1.000000
105,EDWARD STEPHENS,"Bids, Nationals seeding, and big time bragging...","March 30, 2023",Easterns 2023: Tournament Preview & Streaming ...,0.525665
146,ULTIWORLD,"Your guide to the biggest players, teams, and ...","February 3, 2023",D-I Men’s College Primer 2023,0.520078
129,EDWARD STEPHENS,The most stacked D-I men's field yet faces off...,"March 2, 2023",Smoky Mountain Invite 2023: Seven Burning Ques...,0.477491
88,EDWARD STEPHENS,"9 bids are on the line this weekend!APRIL 28, ...","April 28, 2023",D-I College Men’s Regionals 2023: Weekend 1 Pr...,0.453625


We find that, according to the cosine similarity of tf-idf, the article most similar to the nationals preview (article 28), is an article written as a preview to one of the most competitive regular season tournaments of the year, called Easterns. The majority of the top 20 teams were there. Lets look at team mentions and if it tells a similar story

In [122]:
just_mentions = df_team_mentions.drop(['author', 'text', 'date', 'title'], axis = 1)

mention_similarity = cosine_similarity(just_mentions)

In [123]:
df_articles['mentions similarity to 28'] = mention_similarity[28]

In [124]:
df_articles.sort_values(by='mentions similarity to 28', ascending=False)

,author,text,date,title,similarity to 28,mentions similarity to 28
28,JAKE THORNE,Get to know the 20 teams competing for a D-I c...,"May 25, 2023",D-I College Championships 2023: Pool Previews ...,1.000000,1.000000
13,EDWARD STEPHENS,The table is set for some heavyweight quarterf...,"May 28, 2023",D-I College Championships 2023: Putting Prequa...,0.311133,0.850138
15,ALEX RUBIN,Counting down the goings on through day two.MA...,"May 28, 2023",D-I College Championships 2023: Day Two’s 3-2-1,0.439351,0.828907
19,JAKE THORNE,"Friday gave us a potential Sunday preview, as ...","May 27, 2023",D-I College Championships 2023: Day One Recap ...,0.303566,0.763334
22,ULTIWORLD,"The best players in the game right now.MAY 26,...","May 26, 2023",The Top 25 D-I Men’s Players in 2023,0.348659,0.740260
...,...,...,...,...,...,...
147,DANIE PROBY,"Canada takes on Santa Barbara!FEBRUARY 2, 2023...","February 2, 2023",Huckin’ Eh: Santa Barbara Baby! SBI Recap and ...,0.056476,0.403436
156,THEO WAN,The USA College Season is Officially here!JANU...,"January 26, 2023",Huckin’ Eh: Bellingham Recap & Santa Barbara P...,0.060301,0.403399
104,KEITH RAYNOR,Who has a shot going into the final weekend of...,"March 31, 2023",D-I College Bidwatch [3/30/23],0.117590,0.403306
6,ALEX RUBIN,UNC Darkside completed their historic three-pe...,"May 30, 2023",2023 D-I College Championships: Darkside Make ...,0.400160,0.401260


Unlike the tf-idf values, team mentions find that the articles most similar to the nationals precursor articles, are other articles about nationals. This is likely because they all mention the majority if not all of the top 20 teams. I hypothesize that the wording used in tournament previews differs enough from other articles that tf-idf would find that particular style/vocabulary similar enough to each other to result in the easterns preview article being more similar than other articles about nationals.